In [1]:
import pandas as pd
import numpy as np
import json
from os import listdir
from os.path import isfile, join
import glob

We have the JSON file with a list of objects structured as such... \
standard_name -> the name that everything will be transformed to \
code -> ISO Name \
alternatives -> what the other options may be \
scheme -> how the alternatives are assessed, options: exact (one of the items has to match exactly), or_in (any one item has to be contained in the string), and_in (all of the items must be contained in the string)

### Functions to Fix the Countries

In [2]:
with open('country_correction.json') as json_file:
    country_corrections = json.load(json_file)['countries']

def fill_empty_correction_schemes():
    for cnt in country_corrections:
        if cnt['alternatives'] == [] and cnt['scheme'] == '':
            cnt['alternatives'] = [cnt['standard_name']]
            cnt['scheme'] = 'or_in'

fill_empty_correction_schemes()

def satisfies_scheme(country, scheme, alternatives):
    if scheme == 'exact':
        return country.lower() in alternatives
    elif scheme == 'all_in':
        return [alt in country.lower() for alt in alternatives] == [True] * len(alternatives)
    else:
        return [alt in country.lower() for alt in alternatives] != [False] * len(alternatives)

def correct_country(country):
    standard_names = [d['standard_name'] for d in country_corrections]
    if country.lower() in standard_names:
        return country.title()
    else:
        country_object = next((country_obj for country_obj in country_corrections if satisfies_scheme(country.lower(), country_obj['scheme'], country_obj['alternatives'])), '')
        return country_object['standard_name'].title() if country_object else ''
    
def get_accompanying_code(country):
    country_object = next((country_obj for country_obj in country_corrections if country.lower() == country_obj['standard_name']), '')
    return country_object['code'] if country_object else ''

with open('exclusion_list.txt', 'r') as file:
    exclusion_lines = file.readlines()
    exclusion_lines = [line.rstrip('\n') for line in exclusion_lines]
exclusion_lines = exclusion_lines[1:]

In [3]:
def country_correct_dataset(df, column):
    df = df.rename(columns = {column: 'country'})

    df = df.query('country not in @exclusion_lines')
    df['country'] = df['country'].map(correct_country)
    df['code'] = df['country'].map(get_accompanying_code)
    df = df.sort_values('country')
    df = df[df['country'] != '']
    return df

### WDI, IMF, and Heritage

In [7]:
list_of_csv_datasets = [
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Adjusted Net National Income per Capita/API_NY.ADJ.NNTY.PC.KD_DS2_en_csv_v2_7018.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Exports of Goods and Services (constant 2015 US)/API_NE.EXP.GNFS.KD_DS2_en_csv_v2_7684.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - GDP Deflator/API_NY.GDP.DEFL.KD.ZG_DS2_en_csv_v2_104.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - GDP per Capita PPP/API_NY.GDP.PCAP.PP.CD_DS2_en_csv_v2_35.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Imports of Goods and Services (constant 2015 US)/API_NE.IMP.GNFS.KD_DS2_en_csv_v2_14087.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Official Exchange Rate/API_PA.NUS.FCRF_DS2_en_csv_v2_58.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Population/API_SP.POP.TOTL_DS2_en_csv_v2_61.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Social Contributions/API_GC.REV.SOCL.CN_DS2_en_csv_v2_12643.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Tariff Rate, Applied, Weighted Mean, All Products/API_TM.TAX.MRCH.WM.AR.ZS_DS2_en_csv_v2_440.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Taxes on Goods and Services (Current LCU)/API_GC.TAX.GSRV.CN_DS2_en_csv_v2_7514.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Taxes on Goods and Services (Percentage VAT)/API_GC.TAX.GSRV.VA.ZS_DS2_en_csv_v2_12285.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Taxes on Income, Profits, and Capital Gains/API_GC.TAX.YPKG.CN_DS2_en_csv_v2_12653.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Taxes on International Trade/API_GC.TAX.INTT.CN_DS2_en_csv_v2_12651.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - WGI Control of Corruption/API_CC.EST_DS2_en_csv_v2_4455.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - WGI Government Effectiveness/API_GE.EST_DS2_en_csv_v2_4300.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - WGI Regulatory Quality/API_RQ.EST_DS2_en_csv_v2_8225.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - WGI Rule of Law/API_RL.EST_DS2_en_csv_v2_5814.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/IMF - World Revenue/dataset_2026-03-23T17_40_40.193148454Z_DEFAULT_INTEGRATION_IMF.FAD_WORLD_3.0.1.csv', 'COUNTRY'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/Index of Economic Freedom/heritage-index-of-economic-freedom-2026-03-12_2236.csv', 'Country'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Extracting PDF Data/size_of_the_shadow_economy.csv', 'Country'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Average Time to Clear Exports Through Customs (days)/API_IC.CUS.DURS.EX_DS2_en_csv_v2_9141.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Import Unit Value Index/API_TM.UVI.MRCH.XD.WD_DS2_en_csv_v2_10710.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - CPI/API_FP.CPI.TOTL.ZG_DS2_en_csv_v2_287.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - Size of Agricultural Sector/API_NV.AGR.TOTL.ZS_DS2_en_csv_v2_1401.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - FDI/API_BX.KLT.DINV.WD.GD.ZS_DS2_en_csv_v2_115540.csv', 'Country Name'],
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WDI - GDP Deflator Mixed Base Years/API_NY.GDP.DEFL.ZS_DS2_en_csv_v2_8603.csv', 'Country Name']
]
list_of_csv_datasets_temp = [
    ['/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Extracting PDF Data/size_of_the_shadow_economy.csv', 'Country']
]

In [8]:
datasets_to_correct = list_of_csv_datasets_temp # Change this to the sum of whichever datasets you want
for dataset_combo in datasets_to_correct:
    dataset = dataset_combo[0]
    column = dataset_combo[1]
    
    df = country_correct_dataset(pd.read_csv(dataset), column).reset_index(drop = True)
    print(dataset)

    df.to_csv(f'{dataset[:-4]}-adjusted.csv', index = False)

/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Extracting PDF Data/size_of_the_shadow_economy.csv


### WITS

In [6]:
exp_folder = glob.glob('/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WITS/EXP/*.xlsx')
rca_folder = glob.glob('/Users/alex/Documents/School/Research/Trade and the Shadow Economy/Data Analysis/Data/WITS/RCA/*.xlsx')
all_wits_files = exp_folder + rca_folder

In [7]:
for dataset in all_wits_files:
    column = 'Reporter Name'
    
    df = pd.read_excel(dataset, sheet_name = 'Product-TimeSeries-Product', engine = 'openpyxl')
    df = df[~df['Reporter Name'].isin(['Belgium-Luxembourg', 'Netherlands Antilles'])]
    df = country_correct_dataset(df, column).reset_index(drop = True)

    df.to_csv(f'{dataset[:-5]}-adjusted.csv', index = False)